In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load the data

data_path = "../data/interim/final_data.csv"

df = pd.read_csv(data_path, parse_dates=["tpep_pickup_datetime"])

In [3]:
df

,tpep_pickup_datetime,region,total_pickups,avg_pickups
0,2016-01-01 00:00:00,0,58,NaN
1,2016-01-01 00:15:00,0,120,58.0
2,2016-01-01 00:30:00,0,149,97.0
3,2016-01-01 00:45:00,0,160,123.0
4,2016-01-01 01:00:00,0,187,140.0
...,...,...,...,...
262075,2016-03-31 22:45:00,29,14,17.0
262076,2016-03-31 23:00:00,29,17,16.0
262077,2016-03-31 23:15:00,29,18,16.0
262078,2016-03-31 23:30:00,29,13,17.0


In [4]:
# shape of the data

df.shape

(262080, 4)

In [6]:
# datatypes

df.dtypes

tpep_pickup_datetime    datetime64[ns]
region                           int64
total_pickups                    int64
avg_pickups                    float64
dtype: object

In [7]:
# check for missing values

df.isna().sum()

tpep_pickup_datetime    0
region                  0
total_pickups           0
avg_pickups             1
dtype: int64

In [8]:
# extract the day of week information
df["day_of_week"] = df["tpep_pickup_datetime"].dt.day_of_week

# extract the month information
df["month"] = df["tpep_pickup_datetime"].dt.month

In [9]:
# set the datetime column as index

df.set_index("tpep_pickup_datetime", inplace=True)

In [10]:
df

,region,total_pickups,avg_pickups,day_of_week,month
tpep_pickup_datetime,,,,,
2016-01-01 00:00:00,0,58,NaN,4,1
2016-01-01 00:15:00,0,120,58.0,4,1
2016-01-01 00:30:00,0,149,97.0,4,1
2016-01-01 00:45:00,0,160,123.0,4,1
2016-01-01 01:00:00,0,187,140.0,4,1
...,...,...,...,...,...
2016-03-31 22:45:00,29,14,17.0,3,3
2016-03-31 23:00:00,29,17,16.0,3,3
2016-03-31 23:15:00,29,18,16.0,3,3


In [11]:
# create the region grouper

region_grp = df.groupby("region")

region_grp

In [12]:
# shifting periods

periods = list(range(1,5))

periods

[1, 2, 3, 4]

In [13]:
# generate the lag features

lag_features = region_grp["total_pickups"].shift(periods)

lag_features

,total_pickups_1,total_pickups_2,total_pickups_3,total_pickups_4
tpep_pickup_datetime,,,,
2016-01-01 00:00:00,NaN,NaN,NaN,NaN
2016-01-01 00:15:00,58.0,NaN,NaN,NaN
2016-01-01 00:30:00,120.0,58.0,NaN,NaN
2016-01-01 00:45:00,149.0,120.0,58.0,NaN
2016-01-01 01:00:00,160.0,149.0,120.0,58.0
...,...,...,...,...
2016-03-31 22:45:00,22.0,14.0,15.0,13.0
2016-03-31 23:00:00,14.0,22.0,14.0,15.0
2016-03-31 23:15:00,17.0,14.0,22.0,14.0


In [14]:
# merge them with the original df

data = pd.concat([lag_features,df],axis=1)

data

,total_pickups_1,total_pickups_2,total_pickups_3,total_pickups_4,region,total_pickups,avg_pickups,day_of_week,month
tpep_pickup_datetime,,,,,,,,,
2016-01-01 00:00:00,NaN,NaN,NaN,NaN,0,58,NaN,4,1
2016-01-01 00:15:00,58.0,NaN,NaN,NaN,0,120,58.0,4,1
2016-01-01 00:30:00,120.0,58.0,NaN,NaN,0,149,97.0,4,1
2016-01-01 00:45:00,149.0,120.0,58.0,NaN,0,160,123.0,4,1
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4,1
...,...,...,...,...,...,...,...,...,...
2016-03-31 22:45:00,22.0,14.0,15.0,13.0,29,14,17.0,3,3
2016-03-31 23:00:00,14.0,22.0,14.0,15.0,29,17,16.0,3,3
2016-03-31 23:15:00,17.0,14.0,22.0,14.0,29,18,16.0,3,3


In [15]:
print("The shape of the df before merger ", df.shape)
print("The shape of the df after merger ", data.shape)

The shape of the df before merger  (262080, 5)
The shape of the df after merger  (262080, 9)


In [16]:
# rows having missing values

data.isna().any(axis=1).sum()

np.int64(120)

In [18]:
# drop the missing values

data.dropna(inplace=True)

In [19]:
data.isna().any(axis=1).sum()

np.int64(0)

In [20]:
mapper = {name:f"lag_{ind+1}" for ind, name in enumerate(data.columns[0:4])}

mapper

{'total_pickups_1': 'lag_1',
 'total_pickups_2': 'lag_2',
 'total_pickups_3': 'lag_3',
 'total_pickups_4': 'lag_4'}

In [21]:
# replace the column names

data = data.rename(columns=mapper)

In [22]:
data.head()

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week,month
tpep_pickup_datetime,,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4,1
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4,1
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4,1
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4,1
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4,1


In [23]:
# shape of the data

data.shape

(261960, 9)

In [24]:
# number of rows in each month

data['month'].value_counts()

month
3    89280
1    89160
2    83520
Name: count, dtype: int64

In [25]:
data

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week,month
tpep_pickup_datetime,,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4,1
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4,1
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4,1
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4,1
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4,1
...,...,...,...,...,...,...,...,...,...
2016-03-31 22:45:00,22.0,14.0,15.0,13.0,29,14,17.0,3,3
2016-03-31 23:00:00,14.0,22.0,14.0,15.0,29,17,16.0,3,3
2016-03-31 23:15:00,17.0,14.0,22.0,14.0,29,18,16.0,3,3


In [26]:
data.loc[data["month"].isin([1,2]),"lag_1":"day_of_week"]

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,12.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [27]:
# split the data

trainset = data.loc[data["month"].isin([1,2]),"lag_1":"day_of_week"]

testset = data.loc[data["month"].isin([3]),"lag_1":"day_of_week"]

In [28]:
trainset 

,lag_1,lag_2,lag_3,lag_4,region,total_pickups,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,187,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,194,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,180,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,197,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185,185.0,4
...,...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,17,12.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,15,14.0,0


In [ ]:
# save the train and test data

train_data_save_path = "../data/processed/train.csv"

test_data_save_path = "../data/processed/test.csv"

trainset.to_csv(train_data_save_path, index=True)
testset.to_csv(test_data_save_path, index=True)

In [32]:
# make X_train and y_train

X_train = trainset.drop(columns=["total_pickups"])

y_train = trainset["total_pickups"]

In [33]:
X_train.shape

(172680, 7)

In [34]:
X_train

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-01-01 01:00:00,160.0,149.0,120.0,58.0,0,140.0,4
2016-01-01 01:15:00,187.0,160.0,149.0,120.0,0,161.0,4
2016-01-01 01:30:00,194.0,187.0,160.0,149.0,0,175.0,4
2016-01-01 01:45:00,180.0,194.0,187.0,160.0,0,177.0,4
2016-01-01 02:00:00,197.0,180.0,194.0,187.0,0,185.0,4
...,...,...,...,...,...,...,...
2016-02-29 22:45:00,15.0,9.0,11.0,11.0,29,12.0,0
2016-02-29 23:00:00,12.0,15.0,9.0,11.0,29,12.0,0
2016-02-29 23:15:00,17.0,12.0,15.0,9.0,29,14.0,0


In [35]:
y_train

tpep_pickup_datetime
2016-01-01 01:00:00    187
2016-01-01 01:15:00    194
2016-01-01 01:30:00    180
2016-01-01 01:45:00    197
2016-01-01 02:00:00    185
                      ... 
2016-02-29 22:45:00     12
2016-02-29 23:00:00     17
2016-02-29 23:15:00     15
2016-02-29 23:30:00     15
2016-02-29 23:45:00     12
Name: total_pickups, Length: 172680, dtype: int64

In [36]:
# make X_test and y_test

X_test = testset.drop(columns=["total_pickups"])

y_test = testset["total_pickups"]

In [37]:
X_test.shape

(89280, 7)

In [38]:
X_test

,lag_1,lag_2,lag_3,lag_4,region,avg_pickups,day_of_week
tpep_pickup_datetime,,,,,,,
2016-03-01 00:00:00,36.0,44.0,31.0,29.0,0,38.0,1
2016-03-01 00:15:00,41.0,36.0,44.0,31.0,0,39.0,1
2016-03-01 00:30:00,35.0,41.0,36.0,44.0,0,37.0,1
2016-03-01 00:45:00,47.0,35.0,41.0,36.0,0,41.0,1
2016-03-01 01:00:00,34.0,47.0,35.0,41.0,0,38.0,1
...,...,...,...,...,...,...,...
2016-03-31 22:45:00,22.0,14.0,15.0,13.0,29,17.0,3
2016-03-31 23:00:00,14.0,22.0,14.0,15.0,29,16.0,3
2016-03-31 23:15:00,17.0,14.0,22.0,14.0,29,16.0,3


In [39]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_percentage_error

In [40]:
from sklearn import set_config

set_config(transform_output="pandas")

In [43]:

encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(drop="first", sparse_output=False), ["region", "day_of_week"])
], remainder="passthrough", n_jobs=-1)

In [44]:
encoder

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``

In [49]:
# encode the train and test data

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [50]:
# train the model

lr = LinearRegression()

# fit on the training data
lr.fit(X_train_encoded, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](40,)","[ 6.59,-2.06, 1.61,..., 0.28, 0.19,-1.42]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](40,)","['ohe__region_1','ohe__region_2','ohe__region_3',...,'remainder__lag_3', 'remainder__lag_4','remainder__avg_pickups']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.675
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,40
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,40


In [51]:
# make predictions on the train data

y_pred_train = lr.predict(X_train_encoded)

In [52]:
# make predictions on the test data

y_pred_test = lr.predict(X_test_encoded)

In [54]:
# evaluate the baseline model

train_mape = mean_absolute_percentage_error(y_train, y_pred_train)

test_mape = mean_absolute_percentage_error(y_test, y_pred_test)

In [55]:
test_mape

0.30698710068878476

In [56]:
print(f"MAPE on trainset is {(train_mape * 100):.2f}%")
print(f"MAPE on testset is {(test_mape * 100):.2f}%")

MAPE on trainset is 31.75%
MAPE on testset is 30.70%
